# 05.07 Final Model Evaluation — Untouched Authentic Patient-Disjoint Test Set

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay

RANDOM_SEED = 42

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
TRAINED_DIR = PROJECT_ROOT / "models" / "trained"
VIZ_DIR = PROJECT_ROOT / "reports" / "phase_05" / "visualizations" / "final_model_evaluation"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

split = json.loads((MODELS_DIR / "phase05_split_and_schema.json").read_text(encoding="utf-8"))
comp = json.loads((TRAINED_DIR / "model_comparison_summary.json").read_text(encoding="utf-8"))
df_auth = pd.read_csv(PROJECT_ROOT / split["dataset_authentic"])

FEATURE_COLS = split["feature_cols"]
df_test = df_auth.iloc[split["auth_test_indices"]].reset_index(drop=True)
X_test = df_test[FEATURE_COLS].copy()
y_test = df_test["target"].to_numpy(dtype=int)

# Final patient-disjointness assertion
tr_pts = set(df_auth.iloc[split["auth_train_indices"]]["patient_group_id"])
te_pts = set(df_test["patient_group_id"])
assert len(tr_pts & te_pts) == 0, "Patient overlap detected in final test set!"
print(f"Unsealed Untouched Authentic Test Set: N={len(y_test)} unique patients | ELIGIBLE={y_test.sum()}, INELIGIBLE={(1-y_test).sum()}")

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_t = np.asarray(y_true, dtype=int)
    y_p = np.clip(np.asarray(y_prob, dtype=float), 1e-7, 1.0 - 1e-7)
    y_pred = (y_p >= threshold).astype(int)
    cm = confusion_matrix(y_t, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    bal_acc = 0.5 * (spec + rec)
    bins = np.linspace(0.0, 1.0, 11)
    ece = 0.0
    for i in range(10):
        m = (y_p >= bins[i]) & (y_p < bins[i + 1] if i < 9 else y_p <= bins[i + 1])
        if m.any():
            ece += (m.sum() / len(y_t)) * abs(y_t[m].mean() - y_p[m].mean())
    auc = float(roc_auc_score(y_t, y_p)) if len(np.unique(y_t)) == 2 else float("nan")
    return {
        "accuracy": float(accuracy_score(y_t, y_pred)),
        "precision": float(precision_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "recall": float(rec),
        "specificity": float(spec),
        "balanced_accuracy": float(bal_acc),
        "f1": float(f1_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "macro_f1": float(f1_score(y_t, y_pred, average="macro", zero_division=0)),
        "roc_auc": auc,
        "brier_score": float(brier_score_loss(y_t, y_p)),
        "log_loss": float(log_loss(y_t, y_p, labels=[0, 1])),
        "ece": float(ece),
        "confusion_matrix": cm.tolist(),
    }

def bootstrap_ci(y_t, y_p, metric_fn, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_t), size=len(y_t))
        if len(np.unique(y_t[idx])) < 2:
            continue
        vals.append(metric_fn(y_t[idx], y_p[idx]))
    return [round(float(np.percentile(vals, 2.5)), 4), round(float(np.percentile(vals, 97.5)), 4)] if vals else [None, None]

# Load Track A & Track B final models (fitted strictly on train_dev)
lr_a = joblib.load(MODELS_DIR / "logistic_regression_baseline.joblib")
cb_a = joblib.load(TRAINED_DIR / "gradient_boosting" / "catboost_model.joblib")
ebm_a = joblib.load(TRAINED_DIR / "interpretable" / "ebm_model.joblib")

lr_b = joblib.load(MODELS_DIR / "logistic_regression_trackB.joblib")
cb_b = joblib.load(TRAINED_DIR / "gradient_boosting" / "catboost_model_trackB.joblib")
ebm_b = joblib.load(TRAINED_DIR / "interpretable" / "ebm_model_trackB.joblib")

probs_a = {
    "Logistic Regression": lr_a.predict_proba(X_test)[:, 1],
    "CatBoost": cb_a.predict_proba(X_test)[:, 1],
    "Explainable Boosting Machine (EBM)": ebm_a.predict_proba(X_test)[:, 1],
}
probs_a["Soft-Voting Ensemble"] = (probs_a["Logistic Regression"] + probs_a["CatBoost"] + probs_a["Explainable Boosting Machine (EBM)"]) / 3.0

probs_b = {
    "Logistic Regression": lr_b.predict_proba(X_test)[:, 1],
    "CatBoost": cb_b.predict_proba(X_test)[:, 1],
    "Explainable Boosting Machine (EBM)": ebm_b.predict_proba(X_test)[:, 1],
}
probs_b["Soft-Voting Ensemble"] = (probs_b["Logistic Regression"] + probs_b["CatBoost"] + probs_b["Explainable Boosting Machine (EBM)"]) / 3.0

results_a = {}
results_b = {}
summary_rows = []
for mname in probs_a:
    ma = compute_binary_metrics(y_test, probs_a[mname])
    ma["macro_f1_95ci"] = bootstrap_ci(y_test, probs_a[mname], lambda yt, yp: f1_score(yt, (yp >= 0.5).astype(int), average="macro", zero_division=0))
    ma["roc_auc_95ci"] = bootstrap_ci(y_test, probs_a[mname], roc_auc_score)
    mb = compute_binary_metrics(y_test, probs_b[mname])
    results_a[mname] = ma
    results_b[mname] = mb
    summary_rows.append({
        "Model": mname,
        "TrackA_Acc": round(ma["accuracy"], 4),
        "TrackA_MacroF1": round(ma["macro_f1"], 4),
        "TrackA_MacroF1_95CI": str(ma["macro_f1_95ci"]),
        "TrackA_AUC": round(ma["roc_auc"], 4),
        "TrackA_Brier": round(ma["brier_score"], 4),
        "TrackB_Acc": round(mb["accuracy"], 4),
        "TrackB_MacroF1": round(mb["macro_f1"], 4),
        "TrackB_AUC": round(mb["roc_auc"], 4),
    })

df_final = pd.DataFrame(summary_rows)
df_final.to_csv(VIZ_DIR / "final_test_summary_table.csv", index=False)

champion_name = comp["champion_selection"]["champion_model"]
champ_prob = probs_a[champion_name]

# Save test predictions CSV
pred_df = pd.DataFrame({
    "y_true": y_test,
    "prob_lr_trackA": probs_a["Logistic Regression"],
    "prob_cb_trackA": probs_a["CatBoost"],
    "prob_ebm_trackA": probs_a["Explainable Boosting Machine (EBM)"],
    "prob_ens_trackA": probs_a["Soft-Voting Ensemble"],
    "prob_ens_trackB": probs_b["Soft-Voting Ensemble"],
})
pred_df.to_csv(MODELS_DIR / "final_test_predictions.csv", index=False)

final_report = {
    "evaluation_scope": "UNTOUCHED_AUTHENTIC_PATIENT_DISJOINT_TEST_SET",
    "champion_model": champion_name,
    "test_patient_count": int(len(y_test)),
    "test_class_distribution": {"ELIGIBLE": int(y_test.sum()), "INELIGIBLE": int((1 - y_test).sum())},
    "feature_count": int(len(FEATURE_COLS)),
    "track_a_test_results": results_a,
    "track_b_test_results": results_b,
    "governance_attestation": {
        "zero_patient_overlap": True,
        "zero_synthetic_records_in_test": True,
        "zero_prohibited_pii_or_leakage_features": True,
    },
}
(TRAINED_DIR / "final_test_evaluation_report.json").write_text(json.dumps(final_report, indent=2), encoding="utf-8")
print(df_final.to_string(index=False))

# Champion Final Test Plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, (champ_prob >= 0.5).astype(int), display_labels=["INELIGIBLE", "ELIGIBLE"], ax=axes[0], cmap="Blues")
axes[0].set_title(f"{champion_name} — Final Authentic Test CM (N={len(y_test)})")
RocCurveDisplay.from_predictions(y_test, champ_prob, ax=axes[1]); axes[1].set_title(f"{champion_name} — Final Test ROC")
PrecisionRecallDisplay.from_predictions(y_test, champ_prob, ax=axes[2]); axes[2].set_title(f"{champion_name} — Final Test PR")
plt.tight_layout(); plt.savefig(VIZ_DIR / "graph1_final_confusion_matrix.png", dpi=200); plt.close(fig)
print(f"\nFinal evaluation complete. Champion ({champion_name}) Track A Test Macro-F1: {results_a[champion_name]['macro_f1']:.4f}")


Unsealed Untouched Authentic Test Set: N=24 unique patients | ELIGIBLE=15, INELIGIBLE=9


                             Model  TrackA_Acc  TrackA_MacroF1 TrackA_MacroF1_95CI  TrackA_AUC  TrackA_Brier  TrackB_Acc  TrackB_MacroF1  TrackB_AUC
               Logistic Regression      0.9167          0.9111       [0.7778, 1.0]      0.9407        0.0751      0.9167          0.9111      0.9185
                          CatBoost      0.9167          0.9062       [0.7429, 1.0]      0.9926        0.0558      0.9167          0.9062      0.9630
Explainable Boosting Machine (EBM)      0.9583          0.9545       [0.8373, 1.0]      1.0000        0.0315      0.9167          0.9143      0.9556
              Soft-Voting Ensemble      0.9583          0.9545       [0.8373, 1.0]      0.9852        0.0472      0.9583          0.9545      0.9926



Final evaluation complete. Champion (CatBoost) Track A Test Macro-F1: 0.9062
